# Preprocessing — Predicting Daily Citi Bike Ridership
**TLAB IV, Part 2 — Notebook 2 of 3**

This notebook fixes every problem the EDA found and engineers the features the model needs.
Input: `data/citibike_weather_daily.csv` (raw, from Part 1).
Output: `data/citibike_weather_daily_clean.csv` — the **only** file `model.ipynb` is allowed to touch.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/citibike_weather_daily.csv')
print(df.shape)
df.dtypes

(1610, 10)


ride_date               str
num_rides             int64
avg_duration_min    float64
temp_f              float64
max_temp_f          float64
min_temp_f          float64
wind_speed_knots    float64
precip_in           float64
day_of_week             str
month                 int64
dtype: object

## C1: Fix the Dtypes

E1 found one dtype problem: `ride_date` loaded as a string. Convert it to a proper datetime.
All other columns loaded correctly, so no `pd.to_numeric` calls are needed — but we verify
the result rather than assume it.

In [2]:
df['ride_date'] = pd.to_datetime(df['ride_date'], format='%Y-%m-%d')

# Verify: dtype is datetime64 and date arithmetic now works
print(df['ride_date'].dtype)
print('Span:', df['ride_date'].min().date(), '→', df['ride_date'].max().date())
df.dtypes

datetime64[us]
Span: 2013-07-01 → 2018-05-31


ride_date           datetime64[us]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                    str
month                        int64
dtype: object

## C2: Handle the Coded Missing Values

E2 found exactly one sentinel: `precip_in = 99.99` (NOAA's "missing" code for precipitation)
on **2016-02-11**. Step 1: turn the code into a real `NaN` so it can't masquerade as data.
Step 2: decide — drop the row, or impute?

In [3]:
df['precip_in'] = df['precip_in'].replace(99.99, np.nan)

print('NaNs per column after replacement:')
print(df.isna().sum()[df.isna().sum() > 0])
df[df['precip_in'].isna()]

NaNs per column after replacement:
precip_in    1
dtype: int64


,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month
951,2016-02-11,18748,13.523118,28.1,39.9,23.0,17.3,NaN,Thursday,2


In [4]:
# Decision: impute 0.0 (see justification below)
df['precip_in'] = df['precip_in'].fillna(0.0)

print('Remaining NaNs:', df.isna().sum().sum())
print('precip_in max is now:', df['precip_in'].max())
print('precip_in mean is now:', round(df['precip_in'].mean(), 3), '(was 0.178 with the sentinel)')

Remaining NaNs: 0
precip_in max is now: 4.88
precip_in mean is now: 0.116 (was 0.178 with the sentinel)


### C2 justification: impute 0.0 rather than drop

- **Only 1 of 1,610 rows** is affected, so either choice barely moves the model — but the row's
  other columns (rides, temps, wind, calendar) are perfectly good data. Dropping it throws away
  a valid observation to avoid estimating one cell. Imputing keeps it.
- **Why 0.0 specifically:** (a) the median precipitation across the dataset is exactly 0.0 and
  ~63% of all days are completely dry, so 0 is the single most likely value for any day drawn at
  random; (b) the neighboring days that week (Feb 8–14, 2016) recorded 0.00–0.06 inches — a dry
  cold snap — so a "nearby day" imputation lands at ≈0 as well. Both defensible strategies agree.
- A mean imputation would be *wrong* here: the raw mean (0.178) was inflated by the sentinel
  itself, and even the cleaned mean is dragged up by a few storm days — precipitation is far too
  skewed for mean-filling.

## C3: Encode Day of Week

A linear model can't multiply `'Tuesday'` by a coefficient. One-hot encode `day_of_week`
into indicator columns with `pd.get_dummies`.

In [5]:
dow_dummies = pd.get_dummies(df['day_of_week'], prefix='dow', drop_first=True)
df = pd.concat([df, dow_dummies], axis=1)
print('Dummy columns created:', list(dow_dummies.columns))
df.head(3)

Dummy columns created: ['dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday']


,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month,dow_Monday,dow_Saturday,dow_Sunday,dow_Thursday,dow_Tuesday,dow_Wednesday
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,Monday,7,True,False,False,False,False,False
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,Tuesday,7,False,False,False,False,True,False
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,Wednesday,7,False,False,False,False,False,True


### C3 note: why `drop_first=True`

With all 7 day columns, any 6 of them fully determine the 7th (they always sum to 1) — that's
perfect multicollinearity, the "dummy variable trap." Plain `LinearRegression` will still fit,
but the coefficients lose their clean interpretation. Dropping one day makes the remaining six
coefficients mean something concrete: **"how this day differs from the dropped baseline."**
Alphabetically, `drop_first` drops **Friday**, so every `dow_*` coefficient in the model reads
as *rides relative to a Friday with the same weather*.

## C4: Build a Trend Feature

E4 showed the system roughly doubled between 2013 and 2017 — the model needs to know what year
it is. Build both candidate features, pick one for modeling.

In [6]:
df['year'] = df['ride_date'].dt.year
df['days_since_launch'] = (df['ride_date'] - df['ride_date'].min()).dt.days

df[['ride_date', 'year', 'days_since_launch']].iloc[[0, 1, 800, -1]]

,ride_date,year,days_since_launch
0,2013-07-01,2013,0
1,2013-07-02,2013,1
800,2015-09-09,2015,800
1609,2018-05-31,2018,1795


### C4 choice: `days_since_launch`

Both columns are saved, but the model will use **`days_since_launch`**. As a plain numeric
column, `year` is coarse: it forces growth into identical jumps every January 1st and holds it
flat for 12 months (2013.9 and 2014.1 are "a full year apart" to the model). `days_since_launch`
lets the same single coefficient express **smooth, continuous growth** — which is what the E4
time-series actually looks like. (`year` stays in the clean file for anyone who wants it.)

## C5 (Stretch): Engineer Smarter Features

Each feature gets one sentence tying it to something observed in EDA.

In [7]:
# 1. Squared temperature — E3 showed ridership rises with temperature then PLATEAUS past ~85°F;
#    a squared term is how a linear model bends a straight line into that curve.
df['temp_sq'] = df['temp_f'] ** 2

# 2. Weekend flag — E5 showed Saturday/Sunday sit ~20-25% below weekdays as a bloc.
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# 3. Rained-at-all flag — E3 showed rainy days sit visibly lower, and most of the damage is done
#    by ANY rain (the difference between 0.0 and 0.3 inches matters more than 1.0 vs 1.3).
df['rained'] = (df['precip_in'] > 0).astype(int)

df[['temp_f', 'temp_sq', 'is_weekend', 'rained']].describe().loc[['min', 'max', 'mean']].round(2)

,temp_f,temp_sq,is_weekend,rained
min,8.30,68.89,0.00,0.00
max,92.50,8556.25,1.00,1.00
mean,57.37,3610.51,0.28,0.37


## C6: Save the Clean Dataset

Before saving, drop `avg_duration_min`: **Rule 1** bars it from the model (it's computed from
the very rides we're predicting — ops can't know it in advance), and removing it here makes the
leak impossible downstream. All three temperature columns stay in the *file* — Rule 2 is about
what goes into the *model*, and `model.ipynb` will select exactly one.

In [8]:
df = df.drop(columns=['avg_duration_min'])

out_path = '../data/citibike_weather_daily_clean.csv'
df.to_csv(out_path, index=False)

check = pd.read_csv(out_path, parse_dates=['ride_date'])
print(f'Saved {check.shape[0]} rows x {check.shape[1]} cols to {out_path}')
print('\nFinal columns:')
print(list(check.columns))
print('\nSanity: precip max =', check['precip_in'].max(), '| NaNs =', check.isna().sum().sum())

Saved 1610 rows x 20 cols to ../data/citibike_weather_daily_clean.csv

Final columns:
['ride_date', 'num_rides', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'day_of_week', 'month', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday', 'year', 'days_since_launch', 'temp_sq', 'is_weekend', 'rained']

Sanity: precip max = 4.88 | NaNs = 0


### Preprocessing summary

| EDA problem | Fix applied |
|---|---|
| `ride_date` was a string | converted to `datetime64` |
| `precip_in` sentinel 99.99 (1 row) | → NaN → imputed 0.0 (median; neighbors dry) |
| `day_of_week` was text | one-hot encoded, Friday baseline |
| System growth invisible to model | `days_since_launch` (+ `year` kept as alternate) |
| Curved temperature effect | `temp_sq` available for M6 |
| Weekend bloc effect / rain threshold | `is_weekend`, `rained` flags |
| `avg_duration_min` leak risk (Rule 1) | column dropped |

The raw CSV's job is done — `model.ipynb` loads only the clean file.